In [1]:
# %load 5-7.py
import numpy as np
from sklearn import datasets

# Ä£ÐÍÓëÔ¤´¦Àí
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Ä£ÐÍÑ¡ÔñÓë³¬²ÎÊýµ÷ÓÅ
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score

# ÆÀ¹ÀÖ¸±ê
from sklearn.metrics import classification_report, accuracy_score

# ¸ß¼¶ÓÅ»¯¿â (¿ÉÑ¡)
# import optuna

# ¼ÓÔØÊý¾Ý
iris = datasets.load_iris()
X, y = iris.data, iris.target

# Ê×ÏÈ£¬»®·Ö³ö²âÊÔ¼¯ (20%)£¬Ê£ÏÂµÄ80%×÷ÎªÁÙÊ±¡°ÑµÁ·+ÑéÖ¤¡±¼¯
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# È»ºó£¬´ÓÁÙÊ±¼¯ÖÐÔÙ»®·Ö³öÑéÖ¤¼¯ (Õ¼Ô­Ê¼Êý¾ÝµÄ20%£¬¼´ÁÙÊ±¼¯µÄ25%)
# ×îÖÕ±ÈÀý£º ÑµÁ·¼¯(60%)£¬ ÑéÖ¤¼¯(20%)£¬ ²âÊÔ¼¯(20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp # 0.25 * 0.8 = 0.2
)

print(f"训练集大小:  {X_train.shape[0]}")
print(f"验证集大小: {X_val.shape[0]}")
print(f"测试集大小: {X_test.shape[0]}")

# ´´½¨Ò»¸ö¹ÜµÀ£¬ÏÈ±ê×¼»¯Êý¾Ý£¬ÔÙÓ¦ÓÃSVM
pipe = Pipeline([
    ('scaler', StandardScaler()),  # ±ê×¼»¯
    ('svc', SVC(random_state=42))  # ·ÖÀàÆ÷
])

# ¶¨Òå²ÎÊýÍø¸ñ
# ×¢Òâ£º²ÎÊýÃûÐèÒª¼ÓÉÏ²½ÖèÃû³Æ×÷ÎªÇ°×º£¬ÀýÈç ¡®svc__C¡¯
param_grid = {
    'svc__C': [0.1, 1, 10, 100],           # ÕýÔò»¯Ç¿¶È
    'svc__gamma': ['scale', 'auto', 0.01, 0.1, 1], # ºËº¯ÊýÏµÊý
    'svc__kernel': ['rbf', 'linear', 'poly'] # ºËº¯ÊýÀàÐÍ
}
# ´´½¨GridSearchCV¶ÔÏó
# Ëü½«¶ÔÎÒÃÇÌá¹©µÄX_tempºÍy_temp½øÐÐ5ÕÛ½»²æÑéÖ¤
grid_search = GridSearchCV(
    estimator=pipe,         # ÒªÓÅ»¯µÄÄ£ÐÍ/¹ÜµÀ
    param_grid=param_grid,  # ²ÎÊýËÑË÷¿Õ¼ä
    cv=5,                   # 5ÕÛ½»²æÑéÖ¤
    scoring='accuracy',     # ÆÀ¹ÀÖ¸±ê£º×¼È·ÂÊ
    n_jobs=-1,              # Ê¹ÓÃËùÓÐCPUºËÐÄ²¢ÐÐ¼ÆËã
    verbose=1               # Êä³öÏêÏ¸ÈÕÖ¾
)

# Ö´ÐÐÍø¸ñËÑË÷£¡(ÔÚX_temp, y_tempÉÏ½øÐÐ£¬Ëü°üº¬ÁËÎÒÃÇÖ®Ç°µÄX_trainºÍX_val)
grid_search.fit(X_temp, y_temp)

# Êä³ö×î¼Ñ²ÎÊýºÍ×î¼Ñ½»²æÑéÖ¤·ÖÊý
print("最佳参数组合: ", grid_search.best_params_)
print("最佳交叉验证准确率: ", grid_search.best_score_)

# »ñÈ¡×î¼ÑÄ£ÐÍ
best_model = grid_search.best_estimator_

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

# ¶¨Òå²ÎÊý·Ö²¼¶ø·ÇÍø¸ñ
param_distributions = {
    'svc__C': loguniform(1e-2, 1e2),  # ¶ÔÊý¾ùÔÈ·Ö²¼£¬´Ó0.01µ½100
    'svc__gamma': loguniform(1e-3, 1e1),
    'svc__kernel': ['rbf', 'linear']
}

# ´´½¨RandomizedSearchCV¶ÔÏó£¬Ö¸¶¨Ëæ»ú²ÉÑùµÄ´ÎÊý(n_iter)
random_search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=50,  # Ëæ»ú³¢ÊÔ50×é²ÎÊý
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Ö´ÐÐËÑË÷
random_search.fit(X_temp, y_temp)

# Êä³ö½á¹û
print("最佳参数组合 (随机搜索): ", random_search.best_params_)
print("最佳交叉验证准确率 (随机搜索): ", random_search.best_score_)


# Ê×ÏÈ°²×°: pip install optuna
import optuna

def objective(trial):
    # ½¨Òé²ÎÊý·¶Î§
    c = trial.suggest_float('svc__C', 1e-2, 1e2, log=True)
    gamma = trial.suggest_float('svc__gamma', 1e-3, 1e1, log=True)
    kernel = trial.suggest_categorical('svc__kernel', ['rbf', 'linear'])

    # ÉèÖÃ²ÎÊý
    pipe.set_params(svc__C=c, svc__gamma=gamma, svc__kernel=kernel)

    # Ê¹ÓÃ½»²æÑéÖ¤¼ÆËã·ÖÊý
    score = cross_val_score(pipe, X_temp, y_temp, cv=5, scoring='accuracy', n_jobs=-1).mean()
    return score

# ´´½¨study¶ÔÏó£¬×î´ó»¯×¼È·ÂÊ
study = optuna.create_study(direction='maximize')
# Ö´ÐÐÓÅ»¯£¬³¢ÊÔ100´Î
study.optimize(objective, n_trials=100)

# Êä³ö×î¼Ñ½á¹û
print('最佳试验:')
trial = study.best_trial
print(f'  准确率: {trial.value}')
print(f'  最佳参数: {trial.params}')

# ÓÃ×î¼Ñ²ÎÊýÖØÐÂÑµÁ·×îÖÕÄ£ÐÍ
pipe.set_params(**trial.params)
pipe.fit(X_temp, y_temp)
# ... ºóÐøÔÚ²âÊÔ¼¯ÉÏµÄÆÀ¹ÀÍ¬ÉÏ

# Ê¹ÓÃ×î¼ÑÄ£ÐÍÔÚ²âÊÔ¼¯ÉÏ½øÐÐÔ¤²â
y_test_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"\n模型在测试集上的最终准确率: {test_accuracy:.4f}")
print("\n详细分类报告:")
print(classification_report(y_test, y_test_pred, target_names=iris.target_names))



训练集大小:  90
验证集大小: 30
测试集大小: 30
Fitting 5 folds for each of 60 candidates, totalling 300 fits
最佳参数组合:  {'svc__C': 0.1, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}
最佳交叉验证准确率:  0.975
Fitting 5 folds for each of 50 candidates, totalling 250 fits
最佳参数组合 (随机搜索):  {'svc__C': np.float64(13.826232179369875), 'svc__gamma': np.float64(0.006290644294586149), 'svc__kernel': 'rbf'}
最佳交叉验证准确率 (随机搜索):  0.975


C:\Users\popeye\anaconda3\envs\pattern_recognition\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-12-06 15:30:25,388] A new study created in memory with name: no-name-cb7dbe97-440f-4d95-87f3-6de0a4728106
[I 2025-12-06 15:30:25,462] Trial 0 finished with value: 0.9 and parameters: {'svc__C': 0.0180707207453931, 'svc__gamma': 0.297740439139025, 'svc__kernel': 'linear'}. Best is trial 0 with value: 0.9.
[I 2025-12-06 15:30:25,519] Trial 1 finished with value: 0.8666666666666668 and parameters: {'svc__C': 0.021422514758586496, 'svc__gamma': 0.008771109356815876, 'svc__kernel': 'rbf'}. Best is trial 0 with value: 0.9.
[I 2025-12-06 15:30:25,576] Trial 2 finished with value: 0.9666666666666668 and parameters: {'svc__C': 16.59647946769757, 'svc__gamma': 0.01365887423189272, 'svc__kernel': 'rbf'}. Best is trial 2

最佳试验:
  准确率: 0.9833333333333332
  最佳参数: {'svc__C': 15.384765925734012, 'svc__gamma': 0.004206179747594565, 'svc__kernel': 'rbf'}

模型在测试集上的最终准确率: 0.9333

详细分类报告:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

